# Sentiment labels per sentence, saved for the topic notebook

Willian Angelo · Team 10 · Bank of England employer project (CAM_DS_401) · release `2026-09-30`

**Where the labels come from.** The team's sentiment notebook, `notebooks/rachel/All_in_one/sentiment_classification.ipynb` (branch `rachel`, commit `0c39267` of 5 October 2026, written by Rachel Robinson), labels every sentence with four models: an emotion model (BERT-emotion), FinBERT, FinBERT-tone and a RoBERTa sentiment model. It keeps the labels only in memory, summarises them by quarter, and runs one bank at a time, so no file of labels exists.

**What this notebook does.** It copies that notebook's labelling cells unchanged, runs them on both banks, and saves the labels as one row per sentence with the key the topic notebook (`angelo_a3.ipynb`) joins on. The labels are the sentiment track's own: same input file, same cleaning, same models, same settings. Every copied cell says which cell of the source notebook it comes from. Three things differ, and none of them changes a label:

1. **Loading.** The data comes through the team loader from release `2026-09-30`. The source notebook reads `revised_structured_sentences.csv` from a folder on its author's laptop; that file is identical to the release's `structured_sentences.csv` (same 19,092 rows and values, checked on 5 October).
2. **Both banks in one run.** The source notebook keeps one bank before running the models (its cell 17). Each sentence is labelled on its own, and the list of speaker names used in the cleaning is built before that filter, so a sentence gets the same label either way. The bank filter is therefore left out.
3. **GPU.** `device=DEVICE` is added to the four `pipeline(...)` calls, so the models run on the GPU when there is one. Step 4.1 compares the result with the labels saved in the source notebook's outputs.

The saved file holds keys and labels only, no sentence text. The outputs of this notebook print transcript text: **clear all outputs before every commit** (public repository).

## Environment

Team setup cell with `NAME = "angelo"`, then `EXPORT = "2026-09-30"`. The cells below expect `ROOT`, `DATA`, `EXPORT` and `loading`.

In [1]:
# 0.0 local Windows only, run before the setup cell: pandas loads pyarrow, whose bundled msvcp140.dll
# is older than the one torch needs, and torch then fails with WinError 1114. Harmless in Colab.
import torch

Loads PyTorch before anything else. On Windows, loading pandas first stops PyTorch from starting, so this line only keeps the notebook running; it does no analysis.

In [2]:
# project setup - run first, do not edit except NAME
NAME = "angelo"  # <<< your name

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
               check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}  data {DATA}")

ok  python 3.13.15  pandas 2.2.3  data C:\local_CAM\Emp_Proj\boe-financial-analysis\data


Standard project set-up: finds the project folder, updates the code from the team's shared branch, installs the pinned library versions and checks them. The output confirms Python 3.13 and pandas 2.2.3.

In [3]:
EXPORT = "2026-09-30"  # pin the export this notebook was written against
str_sent = loading.load(DATA, "structured_sentences.csv", EXPORT)

loaded structured_sentences.csv  19092 rows


Loads the sentence file of the 30 September release: 19,092 sentences. It is the same file the sentiment notebook reads from a folder on its author's laptop.

## Settings and data

In [4]:
# 1.0 settings: output folder, device, and the key shared with the topic notebook
try:
    import torch
except OSError as e:
    raise OSError("torch failed to load because pandas loaded first: restart the kernel "
                  "and run the 0.0 cell before the setup cell") from e
import time
import numpy as np
import pandas as pd

for name in ("ROOT", "DATA", "EXPORT", "loading"):
    if name not in globals():
        raise NameError(f"run the Environment cell first: {name} is not defined")
OUT = ROOT / "notebooks" / "angelo" / "outputs" / EXPORT      # outputs/ is kept out of git by .git/info/exclude
if DATA in OUT.parents:
    raise RuntimeError(f"never write inside {DATA}: it holds the read-only release")
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"        # added to the source notebook's pipeline calls
KEY = ["bank", "quarter", "call_type", "position_in_call", "sentence_number"]
pd.set_option("display.max_colwidth", 100)
print(f"release {EXPORT}   device {DEVICE}   outputs {OUT}")

release 2026-09-30   device cuda   outputs C:\local_CAM\Emp_Proj\boe-financial-analysis\notebooks\angelo\outputs\2026-09-30


Sets the output folder (outside the read-only data folder), uses the laptop GPU when there is one (here there is), and defines the key that identifies a sentence in every release: bank, quarter, call type, position in call and sentence number.

In [5]:
# 1.1 the source notebook's cell 5, with the release in place of the file on its author's laptop (identical content)
data = str_sent.copy()
print(data.shape)

(19092, 15)


Copies the sentences into a table called `data`, the name the sentiment notebook uses, so its cells below run unchanged: 19,092 sentences and 15 columns.

## Cleaning, copied from the sentiment notebook

These five cells are the source notebook's cells 6, 9, 11, 13 and 14, unchanged. They drop filler sentences, then turn each sentence into a list of words: lower case, letters only, no English stop words, no speaker names, each word reduced to its dictionary form. The models in the next section read these word lists, joined back into a string, not the original sentence.

In [6]:
# copied from the source notebook, cell 6 (unchanged)
# Removerows where is_filler_sentence = True
data = data[data['is_filler_sentence'] != True]
print(data.shape)

(16513, 15)


Drops the sentences the release flags as filler, as the sentiment notebook does. 16,513 sentences are left, from both banks, all sections and all speakers.

In [7]:
# copied from the source notebook, cell 9 (unchanged)
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Willian\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Willian\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Willian\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Willian\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Loads the language tools the cleaning uses: the English stop-word list, the word splitter and the WordNet dictionary that reduces words to their base form. The download lines only check that these files are on the laptop.

In [8]:
# copied from the source notebook, cell 11 (unchanged)
# Create a list using "speaker_name" column from the dataframe
speaker_names = data["speaker_name"].tolist()
# Reduce list to contain only unique speaker names
speaker_names = list(set(speaker_names))
# Split first and last names of the speakers
speaker_names_split = [name.split() for name in speaker_names]
# remove capital letters from speaker names
speaker_names_split = [[name.lower() for name in sublist] for sublist in speaker_names_split]
speaker_names_split[:10]

[['jeremy', 'sigee'],
 ['ralph', 'hamers'],
 ['alastair', 'ryan'],
 ['vishal', 'shah'],
 ['operator'],
 ['sergio', 'p.', 'ermotti'],
 ['gerard', 'cassidy'],
 ['betsy', 'l.', 'graseck'],
 ['kian', 'abouhossein'],
 ['adam', 'terelak']]

Builds the list of speaker names, split into first and last names in lower case, so the cleaning can remove them from the text. The list is built from both banks before any bank filter, exactly as in the sentiment notebook.

In [9]:
# copied from the source notebook, cell 13 (unchanged)
def preprocess_text(text):
    if not isinstance(text, str): # Handle non-string input, e.g., NaN
        return []
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize the text
    tokens = word_tokenize(text)

    # Remove stopwords and lemmatize
    stop_words = set(stopwords.words('english'))

    # Remove speaker names from the text
    speaker_names_flat = [name for sublist in speaker_names_split for name in sublist]

    lemmatizer = WordNetLemmatizer()
    cleaned_tokens = [
        lemmatizer.lemmatize(word) for word in tokens
        # Removed "greeting words" from the line below:
        if word not in stop_words and word.isalpha() and word and word not in speaker_names_flat # Ensure only alphabetic words are kept and remove greeting words and speaker names
    ]
    return [word for word in cleaned_tokens if word not in stop_words]

No output: defines the sentiment notebook's cleaning function. It lowercases the text, keeps letters only (numbers and punctuation go), splits it into words, removes English stop words and speaker names, and reduces each word to its dictionary form. The stop-word list includes "not", "no" and direction words such as "up" and "down", so negations and numbers never reach the models.

In [10]:
# copied from the source notebook, cell 14 (unchanged), then the shape its cell 15 shows: (16513, 16)
# Apply the preprocessing function
data['cleaned_text'] = data['sentence'].apply(preprocess_text)

# Display the first few rows
print(data[['sentence', 'cleaned_text']].head(10))
print(data.shape)

                                                                                               sentence  \
1                                         Welcome to JPMorgan Chase's First Quarter 2022 Earnings Call.   
2                                                                          This call is being recorded.   
3                                                 Your line will be muted for the duration of the call.   
4                                                              We will now go live to the presentation.   
6   At this time, I would like to turn the call over to JPMorgan Chase's Chairman and CEO, Jamie Dim...   
7                                                                          Mr. Barnum, please be ahead.   
10         The presentation is available on our website and please refer to the disclaimer in the back.   
11  Starting on page 1, the firm reported net income of $8.3 billion, EPS of $2.63, on revenue of $3...   
12  These results include approximate

Applies the cleaning to every sentence. The sample shows what the models read: "Your line will be muted for the duration of the call." becomes [line, muted, duration, call]. The table now has 16,513 rows and 16 columns, the same shape the sentiment notebook shows at this point.

## The four models, copied from the sentiment notebook

Each model has two cells, as in the source notebook: one loads the model and builds the list of texts, one runs it and adds the labels to `data`. The only change is `device=DEVICE` in the `pipeline(...)` call; a timing line is added at the start and end of each run. The source notebook's comments still say "UBS" because it ran one bank at a time; here every sentence of both banks is labelled.

| Model | Hugging Face name | Columns added to `data` |
|---|---|---|
| BERT-emotion | `bhadresh-savani/bert-base-uncased-emotion` | `main_emotion`, `main_emotion_score`, one score per emotion (`anger`, `joy`, `fear`, `sadness`, `love`, `surprise`) |
| FinBERT | `prosusai/finbert` | `main_sentiment`, `main_sentiment_score`, `positive`, `negative`, `neutral` |
| FinBERT-tone | `yiyanghkust/finbert-tone` | `FBT_Main_Sentiment`, `FBT_Main_Sentiment_Score`, `FBT_Positive`, `FBT_Negative`, `FBT_Neutral` |
| RoBERTa | `cardiffnlp/twitter-roberta-base-sentiment` | `RBT_main_sentiment`, `RBT_main_sentiment_score`, `RBT_negative`, `RBT_neutral`, `RBT_positive` |

In [11]:
# copied from the source notebook, cell 26; only change: device=DEVICE
# Conduct Bert Emotion analysis on cleaned sentences from presentation section (UBS)
from transformers import pipeline
emotion_analyzer = pipeline("text-classification", model="bhadresh-savani/bert-base-uncased-emotion", device=DEVICE)
documents = (
    data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

C:\local_CAM\Emp_Proj\boe-financial-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4159.60it/s]

Loads the emotion model, a BERT model trained to tag six emotions, on the GPU, and joins each cleaned word list back into one string for it.

In [12]:
# copied from the source notebook, cell 28 (unchanged apart from the timing lines)
t0 = time.time()  # added: timing
# Run the model and keep all emotion scores
basic_emotion = emotion_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
emotion_rows = []
for result in basic_emotion:
    if isinstance(result, list):
        emotion_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        emotion_rows.append({result["label"]: result["score"]})
    else:
        emotion_rows.append({})

# Add the top emotion and score to the dataframe
data["main_emotion"] = [
    max(row, key=row.get, default="neutral") if row else "neutral"
    for row in emotion_rows
]
data["main_emotion_score"] = [
    row.get(max(row, key=row.get, default="neutral"), 0.0) if row else 0.0
    for row in emotion_rows
]

# Add all emotion columns as separate numeric columns
for emotion in ["anger", "joy", "fear", "sadness", "love", "surprise"]:
    data[emotion] = [row.get(emotion, 0.0) for row in emotion_rows]
print(f"{len(documents):,} sentences labelled in {time.time() - t0:.0f}s on {DEVICE}")  # added: timing

16,513 sentences labelled in 255s on cuda


Runs the emotion model on all 16,513 sentences (about 4 minutes on the GPU) and adds to every sentence its main emotion, that emotion's score and a score for each of the six emotions.

In [13]:
# copied from the source notebook, cell 43; only change: device=DEVICE
# Conduct sentiment analysis using FinBERT on UBS-related financial texts
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

# Load FinBERT model and tokenizer
model_name = "prosusai/finbert"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Create sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=DEVICE)

documents = (
    data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 27959.24it/s]

Loads FinBERT, a BERT model trained on financial news to tag a text as positive, negative or neutral, and prepares the same strings.

In [14]:
# copied from the source notebook, cell 45 (unchanged apart from the timing lines)
t0 = time.time()  # added: timing
# Run the model and keep all sentiment scores
basic_sentiment = sentiment_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
sentiment_rows = []
for result in basic_sentiment:
    if isinstance(result, list):
        sentiment_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        sentiment_rows.append({result["label"]: result["score"]})
    else:
        sentiment_rows.append({})

# Add the top sentiment and score to the dataframe
data["main_sentiment"] = [
    max(row, key=row.get, default="neutral") if row else "neutral"
    for row in sentiment_rows
]
data["main_sentiment_score"] = [
    row.get(max(row, key=row.get, default="neutral"), 0.0) if row else 0.0
    for row in sentiment_rows
]

# Add all sentiment columns as separate numeric columns
for sentiment in ["positive", "negative", "neutral"]:
    data[sentiment] = [row.get(sentiment, 0.0) for row in sentiment_rows]
print(f"{len(documents):,} sentences labelled in {time.time() - t0:.0f}s on {DEVICE}")  # added: timing

16,513 sentences labelled in 265s on cuda


Runs FinBERT on all sentences (about 4 minutes) and adds its main label, that label's score and the three class scores.

In [15]:
# copied from the source notebook, cell 61; only change: device=DEVICE
# Conduct sentiment analysis using FinBERT on UBS-related financial texts
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

# Load FinBERT model and tokenizer
model_name = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Create sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=DEVICE)

documents = (
    data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28240.21it/s]

Loads FinBERT-tone, a FinBERT version trained on analyst reports, with the same three labels written with capital letters.

In [16]:
# copied from the source notebook, cell 63 (unchanged apart from the timing lines), then its cell 65, which
# renames the FinBERT-tone score columns (its trailing data.head(10) left out)
t0 = time.time()  # added: timing
# Run the model and keep all sentiment scores
basic_FBT_sentiment = sentiment_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
sentiment_rows = []
for result in basic_FBT_sentiment:
    if isinstance(result, list):
        sentiment_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        sentiment_rows.append({result["label"]: result["score"]})
    else:
        sentiment_rows.append({})

# Add the top sentiment and score to the dataframe
data["FBT_Main_Sentiment"] = [
    max(row, key=row.get, default="Neutral") if row else "Neutral"
    for row in sentiment_rows
]
data["FBT_Main_Sentiment_Score"] = [
    row.get(max(row, key=row.get, default="Neutral"), 0.0) if row else 0.0
    for row in sentiment_rows
]

# Add all sentiment columns as separate numeric columns
# NOTE: FinBERT-Tone requires capital letters for the sentiment labels
for sentiment in ["Positive", "Negative", "Neutral"]:
    data[sentiment] = [row.get(sentiment, 0.0) for row in sentiment_rows]
print(f"{len(documents):,} sentences labelled in {time.time() - t0:.0f}s on {DEVICE}")  # added: timing

# Update FinBERT-Tone column names to make it clearer
data.rename(
    columns={
        "Positive": "FBT_Positive",
        "Negative": "FBT_Negative",
        "Neutral": "FBT_Neutral",
    },
    inplace=True,
)

16,513 sentences labelled in 269s on cuda


Runs FinBERT-tone (about 4.5 minutes), adds its labels and scores, and renames its score columns to start with FBT_, as the sentiment notebook does.

In [17]:
# copied from the source notebook, cell 80; only change: device=DEVICE
# Conduct sentiment analysis using FinBERT on UBS-related financial texts
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Load RoBERTa model and tokenizer
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=DEVICE)

documents = (
    data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19748.30it/s]

Loads the RoBERTa sentiment model, trained on tweets, whose labels LABEL_0, LABEL_1 and LABEL_2 mean negative, neutral and positive.

In [18]:
# copied from the source notebook, cell 82 (unchanged apart from the timing lines)
t0 = time.time()  # added: timing
# Run the model and keep all sentiment scores
basic_sentiment = sentiment_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
sentiment_rows = []
for result in basic_sentiment:
    if isinstance(result, list):
        sentiment_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        sentiment_rows.append({result["label"]: result["score"]})
    else:
        sentiment_rows.append({})

# Add the top sentiment and score to the dataframe
data["RBT_main_sentiment"] = [
    max(row, key=row.get, default="LABEL_1") if row else "LABEL_1"
    for row in sentiment_rows
]
data["RBT_main_sentiment_score"] = [
    row.get(max(row, key=row.get, default="LABEL_1"), 0.0) if row else 0.0
    for row in sentiment_rows
]

# Add all sentiment columns as separate numeric columns
for sentiment in ["LABEL_0", "LABEL_1", "LABEL_2"]:
    data[sentiment] = [row.get(sentiment, 0.0) for row in sentiment_rows]
data.rename(columns={"LABEL_0": "RBT_negative", "LABEL_1": "RBT_neutral", "LABEL_2": "RBT_positive"}, inplace=True)

# Map main_sentiment to "LABEL_0": "negative","LABEL_1": "neutral" and "LABEL_2": "positive"
label_mapping = {
    "LABEL_0": "RBT_negative",
    "LABEL_1": "RBT_neutral",
    "LABEL_2": "RBT_positive",
}
data["RBT_main_sentiment"] = data["RBT_main_sentiment"].map(label_mapping)
print(f"{len(documents):,} sentences labelled in {time.time() - t0:.0f}s on {DEVICE}")  # added: timing

16,513 sentences labelled in 274s on cuda


Runs RoBERTa (about 4.5 minutes), adds its labels and scores, and renames the label codes to RBT_negative, RBT_neutral and RBT_positive.

## Check and save

In [19]:
# 4.1 does this run give the source notebook's labels? Its saved outputs show the labels of the first UBS
# sentences (cells 29, 46, 64 and 83, scores rounded to 6 decimals). Labels must match exactly, scores to 1e-4.
REFERENCE = {   # sentence_id: label and score per model, read from the source notebook's saved outputs
    33595: {"main_emotion": ("joy", 0.970431), "main_sentiment": ("neutral", 0.876172),
            "FBT_Main_Sentiment": ("Neutral", 0.633788), "RBT_main_sentiment": ("RBT_neutral", 0.585431)},
    33597: {"main_emotion": ("anger", 0.889760), "main_sentiment": ("negative", 0.720507),
            "FBT_Main_Sentiment": ("Neutral", 0.762569), "RBT_main_sentiment": ("RBT_neutral", 0.798390)},
    33598: {"main_sentiment": ("neutral", 0.920784), "FBT_Main_Sentiment": ("Neutral", 0.995665),
            "RBT_main_sentiment": ("RBT_neutral", 0.609302)},
    33599: {"main_sentiment": ("positive", 0.841943), "FBT_Main_Sentiment": ("Positive", 1.000000),
            "RBT_main_sentiment": ("RBT_positive", 0.747497)},
    33600: {"main_sentiment": ("neutral", 0.836954), "FBT_Main_Sentiment": ("Positive", 0.999991),
            "RBT_main_sentiment": ("RBT_positive", 0.880521)},
    33601: {"main_sentiment": ("neutral", 0.875320), "FBT_Main_Sentiment": ("Neutral", 0.770857),
            "RBT_main_sentiment": ("RBT_neutral", 0.771707)},
    33602: {"main_sentiment": ("neutral", 0.877357), "FBT_Main_Sentiment": ("Positive", 0.493373),
            "RBT_main_sentiment": ("RBT_neutral", 0.819947)},
    33603: {"main_sentiment": ("positive", 0.539305), "FBT_Main_Sentiment": ("Positive", 0.892135),
            "RBT_main_sentiment": ("RBT_neutral", 0.539774)},
    33604: {"main_sentiment": ("positive", 0.953584), "FBT_Main_Sentiment": ("Positive", 1.000000),
            "RBT_main_sentiment": ("RBT_positive", 0.834321)},
    33605: {"main_sentiment": ("neutral", 0.763131), "FBT_Main_Sentiment": ("Positive", 0.962873),
            "RBT_main_sentiment": ("RBT_positive", 0.789465)},
}
SCORE = {"main_emotion": "main_emotion_score", "main_sentiment": "main_sentiment_score",
         "FBT_Main_Sentiment": "FBT_Main_Sentiment_Score", "RBT_main_sentiment": "RBT_main_sentiment_score"}
by_id = data.set_index("sentence_id")
rows = []
for sid, models in REFERENCE.items():
    for col, (label, score) in models.items():
        rows.append({"sentence_id": sid, "model column": col, "source label": label, "this run": by_id.at[sid, col],
                     "score difference": abs(by_id.at[sid, SCORE[col]] - score)})
check = pd.DataFrame(rows)
check["same label"] = check["source label"] == check["this run"]
display(check.groupby("model column").agg(compared=("same label", "size"), same_label=("same label", "sum"),
                                          largest_score_difference=("score difference", "max")))
assert check["same label"].all() and (check["score difference"] < 1e-4).all(), "this run differs from the source"
print(f"all {len(check)} saved labels reproduced")

,compared,same_label,largest_score_difference
model column,,,
FBT_Main_Sentiment,10,10,1.394978e-06
RBT_main_sentiment,10,10,4.215508e-07
main_emotion,2,2,8.170700e-07
main_sentiment,10,10,1.098602e-06


all 32 saved labels reproduced


Checks this run against the labels the sentiment notebook saved in its own outputs: 32 labels for 10 UBS sentences across the four models. All 32 match, and the scores differ by less than 0.000002, the size of the rounding difference between a GPU and a laptop processor. The labels saved here are therefore the sentiment track's labels.

In [20]:
# 4.2 what the labels say: share of each label per model, by bank and section (% of sentences), and how often
# the three sentiment models agree that a sentence is negative
LABEL_OF = {"BERT-emotion": "main_emotion", "FinBERT": "main_sentiment", "FinBERT-tone": "FBT_Main_Sentiment",
            "RoBERTa": "RBT_main_sentiment"}
for name, col in LABEL_OF.items():
    print(f"\n{name}: % of sentences per label")
    display((100 * pd.crosstab([data["bank"], data["section"]], data[col], normalize="index")).round(1))

negative = pd.DataFrame({"FinBERT": data["main_sentiment"] == "negative",
                         "FinBERT-tone": data["FBT_Main_Sentiment"] == "Negative",
                         "RoBERTa": data["RBT_main_sentiment"] == "RBT_negative"})
votes = negative.sum(axis=1)
print("sentences called negative by 0, 1, 2 or 3 of the sentiment models:", votes.value_counts().sort_index().to_dict())
print("pairwise agreement on 'negative or not' (%):",
      {f"{a} / {b}": round(100 * (negative[a] == negative[b]).mean(), 1)
       for i, a in enumerate(negative) for b in list(negative)[i + 1:]})


BERT-emotion: % of sentences per label


main_emotion   anger  fear   joy  love  sadness  surprise
bank section                                             
JPM  prepared   13.6   1.8  80.4   0.4      3.8       0.0
     qa         27.0   4.0  65.1   0.4      2.9       0.6
UBS  prepared    8.3   2.1  84.0   0.2      5.4       0.0
     qa         20.6   2.3  73.5   0.3      2.5       0.9


FinBERT: % of sentences per label


main_sentiment  negative  neutral  positive
bank section                               
JPM  prepared       13.2     57.8      29.0
     qa              6.7     83.9       9.4
UBS  prepared       14.1     44.3      41.7
     qa              5.6     80.9      13.6


FinBERT-tone: % of sentences per label


FBT_Main_Sentiment  Negative  Neutral  Positive
bank section                                   
JPM  prepared            9.4     62.6      28.0
     qa                 11.8     71.2      17.0
UBS  prepared            8.3     41.9      49.8
     qa                  7.5     71.7      20.8


RoBERTa: % of sentences per label


RBT_main_sentiment  RBT_negative  RBT_neutral  RBT_positive
bank section                                               
JPM  prepared                2.3         83.8          13.9
     qa                      6.3         79.2          14.5
UBS  prepared                2.7         66.2          31.1
     qa                      2.4         82.7          14.9

sentences called negative by 0, 1, 2 or 3 of the sentiment models: {0: 13863, 1: 1824, 2: 629, 3: 197}
pairwise agreement on 'negative or not' (%): {'FinBERT / FinBERT-tone': np.float64(89.7), 'FinBERT / RoBERTa': np.float64(90.4), 'FinBERT-tone / RoBERTa': np.float64(90.2)}


Summarises the labels. FinBERT calls 13% to 14% of prepared-remarks sentences negative but only 6% to 7% of Q&A sentences, where most sentences come out neutral (81% to 84%). The three sentiment models seldom agree on which sentences are negative: 2,650 sentences are negative for at least one model, 826 for at least two and only 197 for all three. The pairwise agreement of about 90% mostly reflects the many sentences that none of them calls negative.

In [21]:
# 4.3 save one row per sentence: the key, the sentence id, section and role, and every model's label and scores.
# No sentence text, so the file holds no transcript content.
LABEL_COLS = ["main_emotion", "main_emotion_score", "anger", "joy", "fear", "sadness", "love", "surprise",
              "main_sentiment", "main_sentiment_score", "positive", "negative", "neutral",
              "FBT_Main_Sentiment", "FBT_Main_Sentiment_Score", "FBT_Positive", "FBT_Negative", "FBT_Neutral",
              "RBT_main_sentiment", "RBT_main_sentiment_score", "RBT_negative", "RBT_neutral", "RBT_positive"]
labels = data[KEY + ["sentence_id", "section", "speaker_role"] + LABEL_COLS]
assert not labels.duplicated(KEY).any(), "the key must identify one sentence"
labels.to_csv(OUT / "sentiment_labels_sentence.csv", index=False)
print(f"wrote sentiment_labels_sentence.csv: {len(labels):,} sentences x {labels.shape[1]} columns")
display(labels.groupby(["bank", "section"]).size().unstack())

wrote sentiment_labels_sentence.csv: 16,513 sentences x 31 columns


section,prepared,qa
bank,,
JPM,1632,6664
UBS,3610,4607


Saves the labels: one row per sentence (16,513) with the key, the sentence id, section, speaker role and every model's label and scores, 31 columns and no sentence text. The topic notebook reads this file and joins it on the key.

### Summary

The sentiment notebook's labelling code was run unchanged on both banks of the 30 September release, on the laptop GPU: about 18 minutes for four models and 16,513 sentences. The result matches the labels saved in that notebook's own outputs (32 of 32, scores within 0.000002), so these are the sentiment track's labels, now stored per sentence in `notebooks/angelo/outputs/2026-09-30/sentiment_labels_sentence.csv` with the key the topic notebook joins on. In the Q&A, FinBERT calls 6% to 7% of sentences negative, and the three sentiment models agree on few negative sentences (826 with at least two votes, 197 with all three).

The labels inherit the sentiment notebook's cleaning: the models read word lists without stop words, negations or numbers. They are used as given; changing them is the sentiment track's decision.

**To refresh the file** after the sentiment notebook changes: copy its newest labelling cells over the ones here (each cell names its source cell), keep the `device=DEVICE` and timing lines, and run this notebook from top to bottom. Then rerun `angelo_a3.ipynb`.